# Listwise реранкинг через DeepSeek API

**Подход:** RankGPT-стиль listwise reranker. Для каждого запроса берём топ-20 документов из retrieval-runs, подаём LLM в одном промпте, получаем переупорядоченный список ID. Один LLM-вызов на запрос.

**Input runs:** четыре файла `runs/`:
* `rrf_query2doc_rus-scifact.json`, `rrf_query2doc_rus-nfcorpus.json` — baseline RRF + Query2doc
* `rrf_query2doc_csqe_rus-scifact.json`, `rrf_query2doc_csqe_rus-nfcorpus.json` — RRF + Query2doc + CSQE

**Output:** 4 файла переупорядоченных runs + сводная таблица с метриками **NDCG@5, NDCG@10, Recall@5, Recall@10, MAP@5, MAP@10** на двух датасетах × двух input-конфигурациях × {без реранкинга, с реранкингом} = 8 строк.

**Стоимость DeepSeek:** ~625 запросов × 4 input runs = 2 492 LLM-вызова. **Внимание:** реранк на baseline и CSQE даёт **частично пересекающиеся** топ-20 множества, и DeepSeek для них может вернуть один и тот же ответ. Чтобы не дёргать API дважды на одинаковые входы, кэшируем по hash(query, top_20_doc_ids).

**CPU-only.** Никакого GPU/локальных моделей.

## 0. Установка зависимостей

In [ ]:
!pip install -q datasets pytrec-eval-terrier openai tqdm pandas nest_asyncio

In [3]:
from __future__ import annotations

import os
import re
import sys
import json
import time
import hashlib
import asyncio
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
import pytrec_eval

# === Директории ===
ROOT = Path('.')
RUN_DIR = ROOT / 'runs'
RES_DIR = ROOT / 'results'
CACHE_DIR = ROOT / 'cache'
RERANK_CACHE = CACHE_DIR / 'rerank_deepseek'
for d in (RUN_DIR, RES_DIR, CACHE_DIR, RERANK_CACHE):
    d.mkdir(parents=True, exist_ok=True)

# === DeepSeek API ===
# DEEPSEEK_API_KEY_PLACEHOLDER'DEEPSEEK_API_KEY', '')
DEEPSEEK_API_KEY_PLACEHOLDER'

DEEPSEEK_MODEL = 'deepseek-chat'
DEEPSEEK_BASE_URL = 'https://api.deepseek.com'
DEEPSEEK_MAX_CONCURRENCY = 32

if not DEEPSEEK_API_KEY_PLACEHOLDER    print('WARNING: DEEPSEEK_API_KEY не задан')

# === Конфигурация реранкинга ===
RERANK_TOP_K = 20             # сколько документов реранкить
RERANK_DOC_TRUNCATE = 500     # символов на документ в промпте
RERANK_MAX_NEW_TOKENS = 256   # ответ — список из ~20 идентификаторов
RERANK_TEMPERATURE = 0.0      # deterministic

## 1. Положи входные runs в ./runs/

Скопируй 4 файла в `runs/`. Если их нет — ноутбук остановится с понятной ошибкой.

In [4]:
INPUT_CONFIGS = [
    # (config_name, dataset, run_path)
    ('q2d',      'rus-scifact',  RUN_DIR / 'rrf_query2doc_rus-scifact.json'),
    ('q2d',      'rus-nfcorpus', RUN_DIR / 'rrf_query2doc_rus-nfcorpus.json'),
    ('q2d_csqe', 'rus-scifact',  RUN_DIR / 'rrf_query2doc_csqe_rus-scifact.json'),
    ('q2d_csqe', 'rus-nfcorpus', RUN_DIR / 'rrf_query2doc_csqe_rus-nfcorpus.json'),
]

missing = [str(p) for _, _, p in INPUT_CONFIGS if not p.exists()]
if missing:
    print('ОШИБКА: отсутствуют входные runs:')
    for m in missing:
        print(f'  - {m}')
    sys.exit(1)
else:
    print(f'Все 4 входных run-файла на месте.')

Все 4 входных run-файла на месте.


## 2. Загрузка датасетов (корпус + queries + qrels)

In [5]:
def load_beir_like(name: str, qrels_split: str = 'test'):
    corpus_ds = load_dataset(f'kaengreg/{name}', 'corpus')['train']
    queries_ds = load_dataset(f'kaengreg/{name}', 'queries')['train']
    qrels_ds = load_dataset(f'kaengreg/{name}-qrels', 'qrels')[qrels_split]

    corpus = {}
    for row in corpus_ds:
        corpus[str(row['_id'])] = {
            'title': row.get('title') or '',
            'text': row.get('text') or '',
        }

    qrels = defaultdict(dict)
    for row in qrels_ds:
        qrels[str(row['query-id'])][str(row['corpus-id'])] = int(row['score'])

    valid_qids = set(qrels.keys())
    queries = {}
    for row in queries_ds:
        qid = str(row['_id'])
        if qid in valid_qids:
            queries[qid] = {'text': row.get('text') or ''}
    qrels = {q: r for q, r in qrels.items() if q in queries}

    print(f'  {name}: corpus={len(corpus)}, queries={len(queries)}, qrels={len(qrels)}')
    return corpus, queries, dict(qrels)


DATASET_NAMES = ['rus-scifact', 'rus-nfcorpus']
DATASETS = {ds: load_beir_like(ds, 'test') for ds in DATASET_NAMES}

Using the latest cached version of the dataset since kaengreg/rus-scifact couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'queries' at C:\Users\Ksyunia\.cache\huggingface\datasets\kaengreg___rus-scifact\queries\0.0.0\75b33d32f2f13f058d0598d6d78f0c3d3afc03d9 (last modified on Sun Apr 12 00:01:02 2026).
Using the latest cached version of the dataset since kaengreg/rus-scifact-qrels couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'qrels' at C:\Users\Ksyunia\.cache\huggingface\datasets\kaengreg___rus-scifact-qrels\qrels\0.0.0\5e0c312c9fb7304a2dc91ec7fd648b3ace5c329f (last modified on Fri May  1 01:42:22 2026).


  rus-scifact: corpus=5183, queries=300, qrels=300


Using the latest cached version of the dataset since kaengreg/rus-nfcorpus couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'corpus' at C:\Users\Ksyunia\.cache\huggingface\datasets\kaengreg___rus-nfcorpus\corpus\0.0.0\66f87c5cb6e134ab138e9421290d077a85f17846 (last modified on Sun Apr 12 00:52:44 2026).
Using the latest cached version of the dataset since kaengreg/rus-nfcorpus couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'queries' at C:\Users\Ksyunia\.cache\huggingface\datasets\kaengreg___rus-nfcorpus\queries\0.0.0\66f87c5cb6e134ab138e9421290d077a85f17846 (last modified on Sun Apr 12 00:52:46 2026).
Using the latest cached version of the dataset since kaengreg/rus-nfcorpus-qrels couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'qrels' at C:\Users\Ksyunia\.cache\huggingface\datasets\kaengreg___rus-nfcorpus-qrels\qrels\0.0.0\2dc2e7fe5e08187db84eeb1da7c3a19d70508337 (las

  rus-nfcorpus: corpus=3633, queries=323, qrels=323


## 3. Метрики (NDCG/Recall/MAP @5 и @10)

In [6]:
K_VALUES = [5, 10]


def evaluate_run(qrels: dict, run: dict, k_values=K_VALUES):
    metric_strs = set()
    for k in k_values:
        metric_strs.add(f'ndcg_cut.{k}')
        metric_strs.add(f'recall.{k}')
        metric_strs.add(f'map_cut.{k}')
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, metric_strs)
    per_q = evaluator.evaluate(run)
    agg = defaultdict(list)
    for q, m in per_q.items():
        for k, v in m.items():
            agg[k].append(v)
    means = {k: float(np.mean(v)) for k, v in agg.items()}
    out = {}
    for k in k_values:
        out[f'NDCG@{k}'] = means.get(f'ndcg_cut_{k}', 0.0)
        out[f'Recall@{k}'] = means.get(f'recall_{k}', 0.0)
        out[f'MAP@{k}'] = means.get(f'map_cut_{k}', 0.0)
    return out

## 4. DeepSeek API клиент

In [7]:
from openai import AsyncOpenAI


class DeepSeekClient:
    def __init__(self, api_key, base_url=DEEPSEEK_BASE_URL,
                 model=DEEPSEEK_MODEL,
                 max_concurrency=DEEPSEEK_MAX_CONCURRENCY):
        self.client = AsyncOpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.sem = asyncio.Semaphore(max_concurrency)

    async def _one(self, messages, temperature=0.0, max_tokens=512, top_p=1.0):
        async with self.sem:
            for attempt in range(4):
                try:
                    resp = await self.client.chat.completions.create(
                        model=self.model,
                        messages=messages,
                        temperature=temperature,
                        top_p=top_p,
                        max_tokens=max_tokens,
                    )
                    return resp.choices[0].message.content or ''
                except Exception as e:
                    if attempt == 3:
                        print(f'  [deepseek] error after retries: {e}')
                        return ''
                    await asyncio.sleep(2 ** attempt)
        return ''

    async def batch(self, messages_list, **kwargs):
        tasks = [self._one(m, **kwargs) for m in messages_list]
        return await asyncio.gather(*tasks)


def _run_async(coro):
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = None
    if loop and loop.is_running():
        import nest_asyncio
        nest_asyncio.apply()
        return loop.run_until_complete(coro)
    return asyncio.run(coro)


if DEEPSEEK_API_KEY_PLACEHOLDER    _ds_client = DeepSeekClient(DEEPSEEK_API_KEY)
else:
    _ds_client = None
    print('DeepSeek клиент не инициализирован — задайте DEEPSEEK_API_KEY')

## 5. Listwise rerank: промпт и парсинг

**Промпт RankGPT-стиля** (адаптирован на русский):
* в систем-роль вкладываем общую инструкцию
* в user-роль — список 20 документов с условными ID `[1]..[20]` + запрос + формат ответа
* парсим ответ regex'ом по `[N]`; если LLM вернула меньше 20 ID — оставшиеся добавляем в конец в исходном порядке

In [8]:
RANK_SYSTEM = (
    'Ты — поисковая система, которая ранжирует документы по их релевантности '
    'к запросу пользователя.'
)

RANK_USER_TEMPLATE = (
    'Я дам тебе {N} документов, каждый с уникальным идентификатором в квадратных '
    'скобках. Ранжируй их по убыванию релевантности к запросу: "{QUERY}".\n\n'
    '{DOCS}\n\n'
    'Запрос: {QUERY}\n\n'
    'Выдай ранжирование документов по убыванию релевантности в формате '
    '[ID] > [ID] > ... > [ID]. Включи все {N} идентификаторов. '
    'Не добавляй пояснений или другого текста.'
)


def _format_docs(doc_ids, corpus, truncate=RERANK_DOC_TRUNCATE):
    parts = []
    for i, did in enumerate(doc_ids, start=1):
        d = corpus.get(did, {})
        title = d.get('title', '') or ''
        text = d.get('text', '') or ''
        full = (title + ' ' + text).strip()[:truncate]
        parts.append(f'[{i}] {full}')
    return '\n'.join(parts)


_BRACKET_RE = re.compile(r'\[(\d+)\]')


def _parse_rank_response(raw: str, n: int) -> list[int]:
    """Возвращает список индексов 1..n в порядке, выданном LLM.
    Пропущенные индексы добавляются в конец в исходном порядке.
    Дубликаты убираются (берётся первое появление).
    """
    if not raw:
        return list(range(1, n + 1))
    matches = _BRACKET_RE.findall(raw)
    seen = set()
    order: list[int] = []
    for m in matches:
        idx = int(m)
        if 1 <= idx <= n and idx not in seen:
            seen.add(idx)
            order.append(idx)
    # Добавим пропущенные в исходном порядке
    for i in range(1, n + 1):
        if i not in seen:
            order.append(i)
    return order


def _cache_key(query: str, doc_ids: list[str]) -> str:
    """Стабильный хеш по (query, упорядоченный список doc_ids).
    Если в двух input-runs топ-20 совпадает по содержанию И порядку — будет один
    LLM-вызов на оба, экономия на API."""
    h = hashlib.sha256()
    h.update(query.encode('utf-8'))
    h.update(b'|')
    h.update('|'.join(doc_ids).encode('utf-8'))
    return h.hexdigest()[:16]

## 6. Реранкинг одного run-а

Для каждого qid:
1. Берём топ-20 doc_id из исходного run'а.
2. Формируем промпт.
3. Один LLM-вызов с кэшированием.
4. Парсим порядок → присваиваем новые скоры по убыванию ранга.
5. Документы с позиций 21+ из исходного run'а **отбрасываются** (метрики @5/@10 их не используют).

Кэш — общий для всех 4 input-runs, ключ = hash(query + doc_ids). Если в q2d и q2d_csqe топ-20 идентичен по составу+порядку → 1 LLM-вызов на оба.

In [9]:
async def _rerank_batch(prompts_payload):
    """prompts_payload: list[dict] с ключами {cache_key, messages}.
    Возвращает {cache_key: raw_response}. Использует общий дисковый кэш."""
    out: dict[str, str] = {}
    to_call = []

    # 1) Подтягиваем кэш с диска
    for item in prompts_payload:
        cache_path = RERANK_CACHE / f'{item["cache_key"]}.json'
        if cache_path.exists():
            out[item['cache_key']] = json.loads(cache_path.read_text(encoding='utf-8'))['response']
        else:
            to_call.append(item)

    # 2) Дедуп: одинаковые cache_key вызываем один раз
    seen_keys = set(out.keys())
    unique_calls = []
    for item in to_call:
        if item['cache_key'] not in seen_keys:
            seen_keys.add(item['cache_key'])
            unique_calls.append(item)

    if not unique_calls:
        return out

    # 3) Параллельный вызов API
    print(f'    LLM calls: {len(unique_calls)} (cached: {len(prompts_payload) - len(unique_calls)})')
    messages_list = [item['messages'] for item in unique_calls]
    raws = await _ds_client.batch(
        messages_list,
        temperature=RERANK_TEMPERATURE,
        max_tokens=RERANK_MAX_NEW_TOKENS,
    )

    # 4) Сохраняем в кэш
    for item, raw in zip(unique_calls, raws):
        out[item['cache_key']] = raw
        cache_path = RERANK_CACHE / f'{item["cache_key"]}.json'
        cache_path.write_text(
            json.dumps({'response': raw}, ensure_ascii=False),
            encoding='utf-8',
        )
    return out


def rerank_run(run: dict, queries: dict, corpus: dict,
               top_k: int = RERANK_TOP_K) -> dict:
    """Возвращает новый run, где для каждого qid топ-K документов переупорядочены LLM,
    а позиции 21+ отброшены."""
    if _ds_client is None:
        raise RuntimeError('DEEPSEEK_API_KEY не задан')

    qids = list(run.keys())

    # 1) Готовим промпты для всех qid
    qid_to_doc_ids: dict[str, list[str]] = {}
    payload = []
    for qid in qids:
        if qid not in queries:
            continue
        sorted_docs = sorted(run[qid].items(), key=lambda x: -x[1])[:top_k]
        doc_ids = [d for d, _ in sorted_docs]
        if not doc_ids:
            qid_to_doc_ids[qid] = []
            continue
        qid_to_doc_ids[qid] = doc_ids

        query_text = queries[qid]['text']
        docs_str = _format_docs(doc_ids, corpus)
        user = (RANK_USER_TEMPLATE
                .replace('{N}', str(len(doc_ids)))
                .replace('{QUERY}', query_text)
                .replace('{DOCS}', docs_str))
        messages = [
            {'role': 'system', 'content': RANK_SYSTEM},
            {'role': 'user',   'content': user},
        ]
        payload.append({
            'qid': qid,
            'cache_key': _cache_key(query_text, doc_ids),
            'messages': messages,
            'n_docs': len(doc_ids),
        })

    # 2) Вызываем API (с кэшем + дедупом)
    raws_by_key = _run_async(_rerank_batch(payload))

    # 3) Парсим и собираем новый run
    new_run = {}
    for item in payload:
        qid = item['qid']
        n = item['n_docs']
        doc_ids = qid_to_doc_ids[qid]
        raw = raws_by_key.get(item['cache_key'], '')
        order_idx = _parse_rank_response(raw, n)  # 1-based индексы в исходном списке
        # Новые скоры — по убыванию ранга, чтобы pytrec_eval сортировал правильно
        scores = {}
        for new_rank, src_idx in enumerate(order_idx, start=1):
            doc_id = doc_ids[src_idx - 1]
            scores[doc_id] = float(n - new_rank + 1)  # топ-1 → score = n, топ-n → 1
        new_run[qid] = scores

    # qid'ы, которых не было в queries (не должно случаться, но на всякий случай)
    for qid in qids:
        if qid not in new_run:
            new_run[qid] = {}

    return new_run

## 7. Прогон по 4 input-runs

In [10]:
all_metrics = []

for cfg_name, ds_name, run_path in INPUT_CONFIGS:
    corpus, queries, qrels = DATASETS[ds_name]
    print(f'\n{"=" * 70}\n{cfg_name}  on  {ds_name}\n{"=" * 70}')

    src_run = json.loads(run_path.read_text(encoding='utf-8'))
    print(f'  Loaded {len(src_run)} queries from {run_path.name}')

    # Бейзлайн (без реранкинга) — те же метрики на исходном run
    base_m = evaluate_run(qrels, src_run)
    all_metrics.append({
        'dataset': ds_name, 'retrieval': cfg_name, 'rerank': 'none',
        **base_m,
    })
    print(f'  baseline (no rerank): NDCG@10={base_m["NDCG@10"]:.4f}  '
          f'Recall@10={base_m["Recall@10"]:.4f}  MAP@10={base_m["MAP@10"]:.4f}')

    # Реранкинг
    t0 = time.time()
    new_run = rerank_run(src_run, queries, corpus, top_k=RERANK_TOP_K)
    print(f'  rerank done in {time.time() - t0:.1f}s')

    # Сохраняем переупорядоченный run
    out_path = RUN_DIR / f'rerank_deepseek_{cfg_name}_{ds_name}.json'
    out_path.write_text(json.dumps(new_run, ensure_ascii=False), encoding='utf-8')
    print(f'  saved {out_path}')

    # Метрики после реранка
    rerank_m = evaluate_run(qrels, new_run)
    all_metrics.append({
        'dataset': ds_name, 'retrieval': cfg_name, 'rerank': 'deepseek',
        **rerank_m,
    })
    print(f'  + DeepSeek rerank:    NDCG@10={rerank_m["NDCG@10"]:.4f}  '
          f'Recall@10={rerank_m["Recall@10"]:.4f}  MAP@10={rerank_m["MAP@10"]:.4f}')


q2d  on  rus-scifact
  Loaded 300 queries from rrf_query2doc_rus-scifact.json
  baseline (no rerank): NDCG@10=0.7437  Recall@10=0.8708  MAP@10=0.6970
    LLM calls: 300 (cached: 0)
  rerank done in 18.0s
  saved runs\rerank_deepseek_q2d_rus-scifact.json
  + DeepSeek rerank:    NDCG@10=0.7735  Recall@10=0.8789  MAP@10=0.7364

q2d  on  rus-nfcorpus
  Loaded 323 queries from rrf_query2doc_rus-nfcorpus.json
  baseline (no rerank): NDCG@10=0.3691  Recall@10=0.1807  MAP@10=0.1424
    LLM calls: 323 (cached: 0)
  rerank done in 18.1s
  saved runs\rerank_deepseek_q2d_rus-nfcorpus.json
  + DeepSeek rerank:    NDCG@10=0.4024  Recall@10=0.1925  MAP@10=0.1605

q2d_csqe  on  rus-scifact
  Loaded 300 queries from rrf_query2doc_csqe_rus-scifact.json
  baseline (no rerank): NDCG@10=0.7645  Recall@10=0.8747  MAP@10=0.7240
    LLM calls: 220 (cached: 80)
  rerank done in 13.9s
  saved runs\rerank_deepseek_q2d_csqe_rus-scifact.json
  + DeepSeek rerank:    NDCG@10=0.7783  Recall@10=0.8883  MAP@10=0.7397


## 8. Сводная таблица

In [11]:
df = pd.DataFrame(all_metrics)
metric_cols = ['NDCG@5', 'NDCG@10', 'Recall@5', 'Recall@10', 'MAP@5', 'MAP@10']
for col in metric_cols:
    df[col] = df[col].round(4)
df = df[['dataset', 'retrieval', 'rerank'] + metric_cols]
df.sort_values(['dataset', 'retrieval', 'rerank'], inplace=True)
df.to_csv(RES_DIR / 'rerank_deepseek_summary.csv', index=False)
df

,dataset,retrieval,rerank,NDCG@5,NDCG@10,Recall@5,Recall@10,MAP@5,MAP@10
3,rus-nfcorpus,q2d,deepseek,0.4495,0.4024,0.1586,0.1925,0.1412,0.1605
2,rus-nfcorpus,q2d,none,0.4055,0.3691,0.1415,0.1807,0.1235,0.1424
7,rus-nfcorpus,q2d_csqe,deepseek,0.4521,0.4151,0.1609,0.2031,0.1402,0.1642
6,rus-nfcorpus,q2d_csqe,none,0.4322,0.3941,0.1565,0.1935,0.1324,0.1525
1,rus-scifact,q2d,deepseek,0.7598,0.7735,0.8396,0.8789,0.7293,0.7364
0,rus-scifact,q2d,none,0.7210,0.7437,0.8071,0.8708,0.6852,0.6970
5,rus-scifact,q2d_csqe,deepseek,0.7654,0.7783,0.8519,0.8883,0.7326,0.7397
4,rus-scifact,q2d_csqe,none,0.7494,0.7645,0.8309,0.8747,0.7160,0.7240


In [12]:
# Wide-формат: дельты от реранкинга
rows = []
for (ds_name, ret_name), grp in df.groupby(['dataset', 'retrieval']):
    g = grp.set_index('rerank')
    if 'none' in g.index and 'deepseek' in g.index:
        for col in metric_cols:
            rows.append({
                'dataset': ds_name,
                'retrieval': ret_name,
                'metric': col,
                'no rerank': g.loc['none', col],
                'deepseek rerank': g.loc['deepseek', col],
                'delta': round(g.loc['deepseek', col] - g.loc['none', col], 4),
            })
delta_df = pd.DataFrame(rows)
delta_df.to_csv(RES_DIR / 'rerank_deepseek_deltas.csv', index=False)
delta_df

,dataset,retrieval,metric,no rerank,deepseek rerank,delta
0,rus-nfcorpus,q2d,NDCG@5,0.4055,0.4495,0.0440
1,rus-nfcorpus,q2d,NDCG@10,0.3691,0.4024,0.0333
2,rus-nfcorpus,q2d,Recall@5,0.1415,0.1586,0.0171
3,rus-nfcorpus,q2d,Recall@10,0.1807,0.1925,0.0118
4,rus-nfcorpus,q2d,MAP@5,0.1235,0.1412,0.0177
5,rus-nfcorpus,q2d,MAP@10,0.1424,0.1605,0.0181
6,rus-nfcorpus,q2d_csqe,NDCG@5,0.4322,0.4521,0.0199
7,rus-nfcorpus,q2d_csqe,NDCG@10,0.3941,0.4151,0.0210
8,rus-nfcorpus,q2d_csqe,Recall@5,0.1565,0.1609,0.0044
9,rus-nfcorpus,q2d_csqe,Recall@10,0.1935,0.2031,0.0096
